# Domain 1: Solution Design & Architecture
**Exam**: Claude Certified Architect – Professional (CCAR-P)
**Weighting**: 17% of exam
**Depth**: deep

## Domain Overview

This domain covers translating business problems into Claude-based solutions, designing
end-to-end architectures (input -> processing -> output -> feedback), selecting the right
architectural pattern (workflow, agentic, augmented LLM), designing multi-agent orchestration, and
aligning solutions to business value pillars.

**Why it matters**: Professional-level architects are expected to make and defend high-level design
decisions, not just implement a given spec.

**Key concepts to know**: workflow vs. agentic vs. augmented-LLM patterns, end-to-end architecture
design, multi-agent orchestration strategy, decomposition, and business-value alignment.

In [ ]:
# Setup: a small helper so every cell in this notebook runs end-to-end,
# whether or not you have an ANTHROPIC_API_KEY configured.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    """Calls the real Claude API if ANTHROPIC_API_KEY is set; otherwise returns a
    simulated response so this notebook is runnable without API access."""
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception as e:
        return (f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY and `pip install anthropic` "
                f"to call the real API]\nPrompt received ({len(prompt)} chars): {prompt[:150]}...")

print("call_claude() ready. Set ANTHROPIC_API_KEY as an environment variable to hit the real API.")

## Key Concepts

### Translate business problems into Claude-based AI solutions

The first architectural step is restating a business problem as a solvable AI problem:
what decision or output is needed, what data is available, and what "good" looks like. Skipping this
step leads to technically impressive but business-irrelevant solutions.

In [ ]:
business_problem = "Support tickets take too long to resolve and customers are churning."

def translate_to_ai_solution(problem: str) -> dict:
    return {
        "decision_or_output_needed": "Faster, higher-quality first responses to tickets",
        "available_data": "historical tickets, resolution notes, product docs",
        "success_criteria": "reduce median first-response time by 50% without hurting CSAT",
    }

print(translate_to_ai_solution(business_problem))

### Design end-to-end architectures and select the right pattern

An end-to-end architecture has input -> processing -> output -> feedback loop stages.
Choose a **workflow** (fixed, predictable steps) when the process is well-understood; an **agentic**
pattern (the model decides its own steps) when the path varies by case; an **augmented LLM**
(single call + tools/retrieval, no autonomous looping) for simpler, bounded tasks.

In [ ]:
def choose_architecture_pattern(predictable_steps: bool, needs_autonomous_looping: bool) -> str:
    if needs_autonomous_looping:
        return "agentic - model decides its own next steps"
    if predictable_steps:
        return "workflow - fixed, predictable pipeline"
    return "augmented LLM - single call with tools/retrieval, no looping"

print(choose_architecture_pattern(predictable_steps=True, needs_autonomous_looping=False))
print(choose_architecture_pattern(predictable_steps=False, needs_autonomous_looping=True))

### Design multi-agent systems and orchestration strategies

Multi-agent systems make sense when a task naturally decomposes into specialized roles
(e.g., retrieval, analysis, synthesis) with different tool needs. Orchestration strategy (centralized
coordinator vs. peer-to-peer handoff) should match how much cross-agent coordination and
observability the use case requires.

In [ ]:
def needs_multi_agent(distinct_specialized_roles: int, requires_central_observability: bool) -> str:
    if distinct_specialized_roles >= 2 and requires_central_observability:
        return "multi-agent with a coordinator (hub-and-spoke) for observability and control"
    if distinct_specialized_roles >= 2:
        return "multi-agent with direct handoff between agents"
    return "single agent is sufficient"

print(needs_multi_agent(3, True))
print(needs_multi_agent(1, False))

### Apply decomposition techniques for complex problem solving

Decompose a complex system design into layers: data/knowledge layer, reasoning/agent layer,
tool/integration layer, and presentation/feedback layer. Each layer can be designed and evaluated
somewhat independently, which simplifies both design and later debugging.

In [ ]:
architecture_layers = {
    "data/knowledge": "source documents, embeddings, retrieval index",
    "reasoning/agent": "Claude models, prompts, agent loop, subagents",
    "tool/integration": "MCP servers, APIs, internal systems",
    "presentation/feedback": "UI, human review, logging, evaluation loop",
}
for layer, contents in architecture_layers.items():
    print(f"{layer:22s}: {contents}")

### Align solutions to business value pillars

Every architectural decision should trace back to a business value pillar: efficiency (do
more with the same resources), transformation (enable a new capability), productivity (speed up
people), cost (reduce spend), or performance SLAs (meet a latency/availability target). Naming the
pillar clarifies which tradeoffs are acceptable.

In [ ]:
def primary_value_pillar(goal: str) -> str:
    mapping = {
        "reduce headcount needed for the same volume": "efficiency",
        "launch a product that wasn't possible before": "transformation",
        "help each employee do more per day": "productivity",
        "lower our monthly cloud bill": "cost",
        "guarantee under 2s response time": "performance SLA",
    }
    return mapping.get(goal, "unclear - clarify with stakeholders")

print(primary_value_pillar("reduce headcount needed for the same volume"))

## Mini Hands-On Project

**Covers**: all Domain 1 sub-objectives.

**Scenario**: Design a top-level architecture for a claims-processing assistant at an insurance
company, choosing the right pattern and value pillar.

### Step 1: Translate the business problem

In [ ]:
problem = "Claims adjusters spend 3 hours per claim on manual document review before they can even start assessing it."
print(translate_to_ai_solution(problem))

### Step 2: Choose the architecture pattern and orchestration strategy

In [ ]:
print(choose_architecture_pattern(predictable_steps=False, needs_autonomous_looping=True))
print(needs_multi_agent(distinct_specialized_roles=3, requires_central_observability=True))

### Step 3: Name the business value pillar and decompose into layers

In [ ]:
print(primary_value_pillar("help each employee do more per day"))
print(architecture_layers)

## Lab Exercises

### Exercise 1: Business problem translation drill

**Objective**: Translate 3 vague business asks into structured AI-solution briefs.

**Produce**: 3 structured briefs (decision needed, data available, success criteria).

In [ ]:
asks = [
    "Our sales reps spend too much time writing proposals.",
    "We get too many duplicate support tickets.",
    "Our onboarding docs are outdated and hard to search.",
]
# TODO: write translate_to_ai_solution-style briefs for each
briefs = []
for a in asks:
    briefs.append({})  # TODO
for a, b in zip(asks, briefs):
    print(a, "->", b)

# Expected output: 3 briefs each naming a concrete decision/output, data source, and measurable success criterion.

In [ ]:
# STRETCH GOAL: rank the 3 asks by a rough (value / feasibility) heuristic and justify the order.
ranking = []  # TODO: list of (ask, reasoning)
print(ranking)

# Expected output: A ranked priority order for the 3 asks based on estimated business value and feasibility.

### Exercise 2: Pattern selection for 4 systems

**Objective**: Choose workflow/agentic/augmented-LLM for 4 described systems.

**Produce**: 4 pattern choices with justification.

In [ ]:
systems = [
    "A fixed 5-step invoice approval pipeline",
    "A research assistant that decides its own search strategy per query",
    "A single-call FAQ answering bot using a knowledge base",
    "A code-review agent that decides how many files to inspect based on findings",
]
# TODO: classify each and justify
for s in systems:
    pattern = "..."  # TODO
    reason = "..."  # TODO
    print(s, "->", pattern, "--", reason)

# Expected output: Correct pattern per system, with the invoice pipeline and FAQ bot NOT classified as agentic.

In [ ]:
# STRETCH GOAL: for the one you called "augmented LLM", describe what would need to change
# for it to justify becoming "agentic" in the future.
migration_trigger = "..."  # TODO
print(migration_trigger)

# Expected output: A concrete trigger condition (e.g., growing query diversity) that would justify migrating to agentic.

### Exercise 3: Multi-agent vs. single-agent decision

**Objective**: Decide multi-agent vs. single-agent for 3 scenarios and design the orchestration if multi-agent.

**Produce**: 3 decisions, with an orchestration sketch for the multi-agent ones.

In [ ]:
scenarios = [
    ("A researcher that searches, reads PDFs, and writes a report", 3, True),
    ("A simple FAQ bot", 1, False),
    ("A code migration tool needing planning, execution, and testing roles", 3, True),
]
# TODO: call needs_multi_agent for each, and for multi-agent results, sketch a coordinator + subagent list
for desc, roles, central in scenarios:
    pass

# Expected output: Correct multi-agent decisions with a sensible coordinator + subagent breakdown for the qualifying cases.

In [ ]:
# STRETCH GOAL: for the code migration tool, identify one risk of the multi-agent design and one
# mitigation (e.g., context isolation risk -> explicit context passing).
risk = "..."  # TODO
mitigation = "..."  # TODO

# Expected output: A specific, realistic risk (not generic) and a concrete mitigation tied to it.

### Exercise 4: Layered decomposition for a new system

**Objective**: Decompose a described RAG-based legal research tool into the 4 architecture layers.

**Produce**: A filled-in layer breakdown.

In [ ]:
system_description = "A tool that lets lawyers ask questions against a firm's case history and get cited answers."
# TODO: fill in what belongs in each of the 4 layers for this specific system
layers = {
    "data/knowledge": "...",
    "reasoning/agent": "...",
    "tool/integration": "...",
    "presentation/feedback": "...",
}
print(layers)

# Expected output: A specific (not generic) breakdown naming actual components like a case-law index, citation-aware prompts, etc.

In [ ]:
# STRETCH GOAL: identify which layer is the highest-risk for this system and why.
highest_risk_layer = "..."  # TODO
reason = "..."  # TODO

# Expected output: A reasoned identification of the data/knowledge layer as highest-risk, given legal accuracy requirements.

### Exercise 5: Value pillar alignment memo

**Objective**: Write a short memo aligning a proposed architecture to its primary value pillar and flagging a mismatch if one exists.

**Produce**: A memo identifying the pillar and any architecture/pillar mismatch.

In [ ]:
proposed_architecture = "A complex 4-agent research pipeline with extensive human review at every step, proposed to 'save costs' on a research team."

# TODO: identify the actual value pillar this architecture would deliver, and note if it conflicts
# with the stated goal of "saving costs" (heavy human review often adds cost, not saves it)
actual_pillar = "..."  # TODO
mismatch_note = "..."  # TODO
print(actual_pillar, mismatch_note)

# Expected output: Identifies the mismatch: heavy human-review-at-every-step architectures often serve quality/risk-reduction pillars, not cost reduction.

In [ ]:
# STRETCH GOAL: propose an alternative architecture that would actually align with the stated cost-saving goal.
alternative = "..."  # TODO
print(alternative)

# Expected output: A leaner alternative architecture (e.g., fewer agents, targeted review only on low-confidence cases) that better fits the cost goal.

## Self-Assessment: Solution Design & Architecture

Answer each question, then check the Answers cell below.


**Q1.** What is the risk of skipping the 'translate business problem to AI solution' step before designing an architecture?

**Q2.** When should a workflow pattern be preferred over an agentic pattern?

**Q3.** What factor most strongly suggests a multi-agent architecture is warranted?

**Q4.** Name the four architecture layers used for decomposing a complex Claude-based system.

**Q5.** Why should every architectural decision be traceable to a specific business value pillar?

## Answers


**A1.** The resulting solution may be technically impressive but fail to address what actually matters to the business, or lack measurable success criteria.

**A2.** When the process steps are predictable and well-understood, since a fixed pipeline is simpler and more reliable than autonomous decision-making.

**A3.** The presence of distinct specialized roles/responsibilities that benefit from separate context, tools, or expertise.

**A4.** Data/knowledge, reasoning/agent, tool/integration, and presentation/feedback.

**A5.** Because it clarifies which tradeoffs (cost, latency, complexity) are acceptable and prevents architecture that doesn't actually serve the stated goal.

---
**Next**: [Domain 2: Claude Models, Prompting & Context Engineering](./02-claude-models-prompting-and-context-engineering.ipynb)
**Back**: [README](./README.md)